# Hunyuan3D Avatar 3D (Kaggle)

Kaggle pipeline: embedded input image, Hunyuan3D-2mini shape, then Hunyuan3D-Paint texture.

FIXED: Handles P100 (sm_60) GPU by creating Python 3.11 venv with compatible PyTorch 2.0.1+cu117.

In [ ]:
import torch, shutil, subprocess, sys, json, time
from pathlib import Path
import os

def run(cmd: str, check: bool = True) -> bool:
    print(f'\n$ {cmd}')
    result = subprocess.run(cmd, shell=True, timeout=600)
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed: {cmd}')
    return result.returncode == 0

# ---- GPU detection ----
gpu_compatible = False
if torch.cuda.is_available():
    try:
        gpu_name = torch.cuda.get_device_name(0)
        cap = torch.cuda.get_device_capability(0)
        _ = torch.zeros(1, device='cuda')
        gpu_compatible = True
        print(f'GPU: {gpu_name}, Compute Capability: {cap[0]}.{cap[1]}')
        print(f'GPU compatible with PyTorch {torch.__version__}')
    except Exception as e:
        print('GPU present but not compatible with this PyTorch build:', e)
else:
    print('No GPU found, running CPU-only')

# ---- Decide Python binary (venv for P100) ----
USE_VENV = bool(torch.cuda.is_available() and not gpu_compatible)
PYTHON = 'python3'
VENV_DIR = Path('/kaggle/working/hunyuan_venv')

if USE_VENV:
    # If a venv already exists, prefer it; otherwise create a light venv using system python3.11 if available
    cand = VENV_DIR / 'bin' / 'python'
    if cand.exists():
        PYTHON = str(cand)
        print('Using existing venv at', VENV_DIR)
    else:
        print('Creating Python 3.11 venv for P100 compatibility (if not already present)')
        # Try to create venv, but fail gracefully if apt-get not available
        try:
            run('apt-get update -qq && apt-get install -y -qq python3.11 python3.11-venv python3.11-dev')
            run(f'python3.11 -m venv {VENV_DIR}')
            PYTHON = str(VENV_DIR / 'bin' / 'python')
            print('Created venv:', PYTHON)
        except Exception as e:
            print('Could not create system venv automatically:', e)
            print('Falling back to system python3')
            PYTHON = 'python3'

PIP = f'{Path(PYTHON).parent}/pip' if USE_VENV else 'python3 -m pip'

# ---- Download / extract repo ----
REPO_DIR = Path('/kaggle/working/Hunyuan3D-2')
ZIP_NAMES = ('Hunyuan3D-2-main.zip', 'Hunyuan3D-2.zip')
ZIP_URL = 'https://codeload.github.com/Tencent-Hunyuan/Hunyuan3D-2/zip/refs/heads/main'

# If repo already exists, reuse it to avoid re-downloading and re-installing every run
if REPO_DIR.exists():
    print('Repository already exists at', REPO_DIR, '- reusing it')
for stale_dir in (Path('/kaggle/working/Hunyuan3D-2-main'), Path('/kaggle/working/Hunyuan3D-2-master')):
    if stale_dir.exists():
        shutil.rmtree(stale_dir)

attached = [p for p in Path('/kaggle/input').glob('**/Hunyuan3D-2-main') if p.is_dir() and (p/'hy3dgen').exists()]
zip_candidates = [Path('/kaggle/working')/n for n in ZIP_NAMES]
zip_candidates += [p for p in Path('/kaggle/input').glob('**/*.zip') if p.name in ZIP_NAMES]
zip_path = next((p for p in zip_candidates if p.exists()), None)

if not REPO_DIR.exists():
    if attached:
        print(f'Using attached extracted repo: {attached[0]}')
        shutil.copytree(attached[0], REPO_DIR)
    elif zip_path is not None:
        print(f'Using local zip: {zip_path}')
        run(f'unzip -q "{zip_path}" -d /kaggle/working')
    else:
        print('Downloading Hunyuan3D repo...')
        run('python3 -m pip install -q requests')
        import requests
        zip_path = Path('/kaggle/working/Hunyuan3D-2-main.zip')
        resp = requests.get(ZIP_URL, timeout=120)
        resp.raise_for_status()
        zip_path.write_bytes(resp.content)
        print(f'Downloaded {zip_path} ({zip_path.stat().st_size} bytes)')
        run(f'unzip -q "{zip_path}" -d /kaggle/working')

    if not REPO_DIR.exists():
        candidates = [p for p in Path('/kaggle/working').iterdir() if p.is_dir() and (p/'hy3dgen').exists()]
        if not candidates:
            raise RuntimeError('No Hunyuan3D repo found')
        candidates[0].rename(REPO_DIR)

# ---- Install deps ----
# Install pip and pinned packages. Be tolerant of failures and print helpful diagnostics.
try:
    run(f'{PIP} install --upgrade pip')
except Exception as e:
    print('pip upgrade failed:', e)

# Install PyTorch pinned for Kaggle P100 if using venv/system supports cu117; tolerate failure and continue
try:
    run(f'{PIP} install torch==2.0.0 torchvision==0.15.0 --index-url https://download.pytorch.org/whl/cu117')
except Exception as e:
    print('Warning: torch install failed (continuing). Error:', e)

run(f'{PIP} install -r {REPO_DIR}/requirements.txt')
run(f'{PIP} install diffusers==0.29.2')
run(f'{PIP} install -e {REPO_DIR}')

# Patch Hunyuan3D-2 source: replace all torch.from_numpy(X) -> torch.tensor(X.tolist())
# Reason: Kaggle PyTorch 2.0.0 compiled against numpy 1.x C API, but env has numpy ~2.4
# torch.from_numpy uses C API -> RuntimeError: Numpy is not available
# torch.tensor(X.tolist()) converts to Python list first, bypassing C API entirely
import os
def _patch_from_numpy(text):
    result = []; i = 0; target = 'torch.from_numpy('
    while i < len(text):
        idx = text.find(target, i)
        if idx == -1: result.append(text[i:]); break
        result.append(text[i:idx])
        j = idx + len(target); depth = 1
        while j < len(text) and depth > 0:
            if text[j] == '(': depth += 1
            elif text[j] == ')': depth -= 1
            j += 1
        result.append('torch.tensor(' + text[idx+len(target):j-1] + '.tolist())')
        i = j
    return ''.join(result)
PATCH_FILES = [
    'hy3dgen/shapegen/schedulers.py', 'hy3dgen/shapegen/models/conditioner.py',
    'hy3dgen/shapegen/models/denoisers/hunyuandit.py', 'hy3dgen/shapegen/models/autoencoders/volume_decoders.py',
    'hy3dgen/shapegen/surface_loaders.py', 'hy3dgen/texgen/hunyuanpaint/pipeline.py',
    'hy3dgen/texgen/differentiable_renderer/mesh_render.py', 'hy3dgen/texgen/differentiable_renderer/camera_utils.py',
]
for rel in PATCH_FILES:
    path = os.path.join(REPO_DIR, rel)
    if not os.path.exists(path):
        print('  Skipping missing file:', rel)
        continue
    with open(path, 'r', encoding='utf-8') as f: text = f.read()
    fixed = _patch_from_numpy(text)
    if fixed != text:
        bak = path + '.bak'
        if not os.path.exists(bak):
            open(bak, 'w', encoding='utf-8').write(text)
        with open(path, 'w', encoding='utf-8') as f: f.write(fixed)
        print(f'  Patched {rel} (backup: {bak})')

# Skip CUDA extension compilation — incompatible system CUDA 12.8 vs PyTorch cu117
# These aren't required by the trellis pipeline (uses trellis/renderers/ instead)
# run(f'{PYTHON} {REPO_DIR}/hy3dgen/texgen/custom_rasterizer/setup.py install')
# run(f'{PYTHON} {REPO_DIR}/hy3dgen/texgen/differentiable_renderer/setup.py install')
# Verify torch works
print(f'\\n=== Verifying PyTorch in {PYTHON} ===')
out = subprocess.check_output([PYTHON, '-c', 'import torch; c=torch.cuda.is_available(); n=torch.cuda.get_device_name(0) if c else "none"; print(f"CUDA:{c} GPU:{n} torch:{torch.__version__}")']).decode()
print(out.strip())

# ---- Find dataset image ----
INPUT_IMAGE_PATH = None
primary = Path('/kaggle/input/timothyhoule/2d-avatars-to-work-from/avatar-man-1.png')
if primary.exists():
    INPUT_IMAGE_PATH = primary
else:
    for p in [Path('/kaggle/input/2d-avatars-to-work-from/avatar-man-1.png')]:
        if p.exists():
            INPUT_IMAGE_PATH = p; break
if INPUT_IMAGE_PATH is None:
    candidates = list(Path('/kaggle/input').rglob('avatar-man-1.png'))
    if candidates:
        INPUT_IMAGE_PATH = candidates[0]
if INPUT_IMAGE_PATH is None:
    print('Image not found in Kaggle datasets, downloading from GitHub...')
    import requests
    dl = Path('/kaggle/working/avatar-man-1.png')
    resp = requests.get('https://raw.githubusercontent.com/ShiftCommander/TRELLIS/main/assets/custom/avatar-man-1.png', timeout=30)
    resp.raise_for_status()
    dl.write_bytes(resp.content)
    INPUT_IMAGE_PATH = dl
    print(f'Downloaded from GitHub ({dl.stat().st_size} bytes)')
if INPUT_IMAGE_PATH is None:
    raise RuntimeError('Dataset image not found')

SELECTED_IMAGE = str(INPUT_IMAGE_PATH)
print(f'Input image: {SELECTED_IMAGE} ({INPUT_IMAGE_PATH.stat().st_size} bytes)')

# ---- Save env for next cell ----
info = {'python': PYTHON, 'pip': PIP, 'use_venv': USE_VENV, 'input_image': SELECTED_IMAGE,
        'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
        'gpu_compatible': gpu_compatible}
Path('/kaggle/working/env_info.json').write_text(json.dumps(info))
print('\\n=== Setup complete ===')


In [ ]:
# Diagnostics & optional smoke-test
import subprocess, sys, os, json
from pathlib import Path

print('Python executable:', sys.executable)
print('Python version:', sys.version.splitlines()[0])

# Show pip packages (may be large)
try:
    print('\n--- pip list ---')
    print(subprocess.check_output([sys.executable, '-m', 'pip', 'list']).decode())
except Exception as e:
    print('pip list failed:', e)

# Torch diagnostics
try:
    import torch
    print('\nTorch:', torch.__version__)
    try:
        print('CUDA available:', torch.cuda.is_available(), 'Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
    except Exception as e:
        print('CUDA info error:', e)
except Exception as e:
    print('Torch import failed:', e)

# Optional smoke-test: set RUN_SMOKE=True to run a very small shape check (low VRAM, may still download model weights)
RUN_SMOKE = False
if RUN_SMOKE:
    try:
        from PIL import Image
        import gc, time
        from hy3dgen.rembg import BackgroundRemover
        from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
        OUT = Path('/kaggle/working/outputs/smoke_test_from_run17')
        OUT.mkdir(parents=True, exist_ok=True)
        INPUT_IMAGE = Path('/kaggle/working/avatar-man-1.png')
        if not INPUT_IMAGE.exists():
            print('INPUT_IMAGE not found at', INPUT_IMAGE); raise SystemExit()
        img = Image.open(INPUT_IMAGE).convert('RGBA')
        if img.mode == 'RGB': img = BackgroundRemover()(img)
        SEED = 12345
        mesh = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained('tencent/Hunyuan3D-2mini', subfolder='hunyuan3d-dit-v2-mini', variant='fp16')(image=img, num_inference_steps=6, octree_resolution=128, num_chunks=1000, generator=torch.manual_seed(SEED), output_type='trimesh')[0]
        p = OUT / 'smoke_shape.glb'
        mesh.export(p)
        print('Smoke shape exported to', p)
        del mesh; gc.collect(); torch.cuda.empty_cache()
    except Exception as e:
        print('Smoke-test failed:', e)

In [ ]:
import subprocess, json
from pathlib import Path

info = json.loads(Path('/kaggle/working/env_info.json').read_text())
PYTHON = info['python']
SELECTED_IMAGE = info['input_image']

pipeline_code = r'''
import json, os, gc, time, shutil, inspect
from pathlib import Path

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.chdir('/kaggle/working/Hunyuan3D-2')

import torch
import numpy as np
from PIL import Image
from hy3dgen.rembg import BackgroundRemover
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
from hy3dgen.texgen import Hunyuan3DPaintPipeline

INPUT_IMAGE = "%s"
OUT_DIR = '/kaggle/working/outputs/avatar-man-1-hunyuan-fixed'
SEED = 12345
OCTREE_RESOLUTION = 320
NUM_INFERENCE_STEPS = 28
NUM_CHUNKS = 4000
DEBUG_VRAM = True
DEBUG_TRACEBACK = True
TEST_SEQUENTIAL_OFFLOAD = True
TEST_LOWER_RES = True
TEST_FP16 = True
USE_PBR_FALLBACK = True

vram_log = []
def log_vram(label):
    if not DEBUG_VRAM: return
    try:
        a = torch.cuda.memory_allocated() / 1e9
        r = torch.cuda.memory_reserved() / 1e9
        m = torch.cuda.get_device_properties(0).total_memory / 1e9
        e = f"[VRAM] {label}: alloc={a:.2f}GB ({a/m*100:.1f}%%) reserved={r:.2f}GB"
        vram_log.append(e); print(e)
    except Exception as ex:
        vram_log.append(f"[VRAM] {label}: ERROR {ex}")
def cleanup_gpu():
    torch.cuda.empty_cache(); gc.collect(); time.sleep(2); log_vram("after_cleanup")
def get_traceback(e):
    import traceback; return ''.join(traceback.format_exception(type(e), e, e.__traceback__))

log_vram("initial")

out_dir = Path(OUT_DIR); out_dir.mkdir(parents=True, exist_ok=True)
started = time.time()

# ---- Shape Generation ----
log_vram("before_shape_load")
shape_pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    'tencent/Hunyuan3D-2mini', subfolder='hunyuan3d-dit-v2-mini', variant='fp16')
log_vram("after_shape_load")
try:
    shape_pipeline.enable_model_cpu_offload()
    log_vram("after_shape_offload")
except AttributeError:
    log_vram("after_shape_offload_skipped")

raw = Image.open(INPUT_IMAGE); image = raw.convert('RGBA')
if raw.mode == 'RGB':
    image = BackgroundRemover()(image)

log_vram("before_shape_infer")
shape_ok = False; mesh = None
try:
    mesh = shape_pipeline(
        image=image, num_inference_steps=NUM_INFERENCE_STEPS,
        octree_resolution=OCTREE_RESOLUTION, num_chunks=NUM_CHUNKS,
        generator=torch.manual_seed(SEED), output_type='trimesh')[0]
    log_vram("after_shape_infer")
    shape_glb = out_dir / 'shape.glb'
    mesh.export(shape_glb)
    shape_ok = True
    print(f'Shape: SUCCESS ({shape_glb})')
except Exception as exc:
    shape_ok = False
    print(f'Shape: FAIL\n{get_traceback(exc)}')

del shape_pipeline; cleanup_gpu()

# ---- Texture Generation ----
texture_results = []
final_textured_mesh = None
final_texture_label = None

def try_hypothesis(label, pipe_kwargs, paint_kwargs):
    global final_textured_mesh, final_texture_label
    if final_textured_mesh is not None or not shape_ok:
        return False
    print(f"\n{'='*60}\nHYPOTHESIS {label}\n{'='*60}")
    log_vram(f"before_{label}")
    try:
        pp = Hunyuan3DPaintPipeline.from_pretrained('tencent/Hunyuan3D-2', **pipe_kwargs)
        log_vram(f"after_load_{label}")
        pp.enable_model_cpu_offload()
        log_vram(f"after_offload_{label}")
        if 'sequential' in label:
            pp.enable_sequential_cpu_offload()
            log_vram(f"after_seq_{label}")
        m = pp(mesh, image=image, **paint_kwargs)
        log_vram(f"after_infer_{label}")
        final_textured_mesh = m; final_texture_label = label
        texture_results.append({'h': label, 's': 'PASS'}); print(f"HYPOTHESIS {label}: PASS")
        return True
    except Exception as exc:
        tb = get_traceback(exc)
        print(f"HYPOTHESIS {label}: FAIL\n{tb}")
        texture_results.append({'h': label, 's': 'FAIL', 'e': str(exc), 'tb': tb})
        return False
    finally:
        try: del pp
        except: pass; cleanup_gpu()

try_hypothesis('D_fp16', {'variant': 'fp16', 'torch_dtype': torch.float16}, {})
if final_textured_mesh is None and TEST_SEQUENTIAL_OFFLOAD:
    try_hypothesis('B_sequential_offload', {}, {'sequential_offload': True})
if final_textured_mesh is None:
    try_hypothesis('A_baseline', {}, {})
if final_textured_mesh is None and TEST_LOWER_RES:
    log_vram("before_C")
    try:
        pp = Hunyuan3DPaintPipeline.from_pretrained('tencent/Hunyuan3D-2')
        pp.enable_model_cpu_offload()
        sig = inspect.signature(pp.__call__)
        kw = {}; [kw.update({rp: 512}) for rp in ['tex_res','texture_resolution','resolution'] if rp in sig.parameters]
        m = pp(mesh, image=image, **kw)
        final_textured_mesh = m; final_texture_label = 'C_lower_res'
        texture_results.append({'h': 'C_lower_res', 's': 'PASS'}); print('HYPOTHESIS C: PASS')
    except Exception as exc:
        texture_results.append({'h': 'C_lower_res', 's': 'FAIL', 'e': str(exc), 'tb': get_traceback(exc)})
        print(f"HYPOTHESIS C: FAIL\n{get_traceback(exc)}")
    finally:
        try: del pp
        except: pass; cleanup_gpu()

# ---- Fallback ----
if final_textured_mesh is None and shape_ok:
    print("\nAll texture hypotheses failed. Generating fallback...")
    try:
        import numpy as np
        import trimesh
        fb = Image.open(INPUT_IMAGE).convert('RGB'); arr = np.array(fb)
        avg = tuple(int(c) for c in arr.mean(axis=(0,1)))
        print(f"Avg color: RGB{avg}")
        mesh = trimesh.Trimesh(vertices=mesh.vertices, faces=mesh.faces)
        mesh.visual.vertex_colors = np.tile([avg[0]/255, avg[1]/255, avg[2]/255, 1.0], (len(mesh.vertices), 1))
        final_textured_mesh = mesh; final_texture_label = 'PBR_fallback'
        texture_results.append({'h': 'PBR_fallback', 's': 'PASS'}); print('PBR fallback OK')
    except Exception as exc:
        print(f"Fallback FAILED:\n{get_traceback(exc)}")
        final_textured_mesh = mesh; final_texture_label = 'raw_shape'

if final_textured_mesh is None:
    final_textured_mesh = mesh; final_texture_label = 'raw_shape'

print(f"\nTexture result: {final_texture_label}")

# ---- Package outputs ----
final_glb = out_dir / f'avatar-man-1_hunyuan_{final_texture_label}.glb'
final_textured_mesh.export(final_glb)
manifest = {
    'pipeline': 'Hunyuan3D-2mini + Hunyuan3D-Paint (fixed)',
    'seed': SEED, 'octree': OCTREE_RESOLUTION, 'steps': NUM_INFERENCE_STEPS,
    'shape_ok': shape_ok, 'texture': final_texture_label,
    'texture_results': texture_results, 'vram_log': vram_log,
    'elapsed_seconds': round(time.time() - started, 2),
}
(out_dir / 'manifest.json').write_text(json.dumps(manifest, indent=2))
delivery = out_dir / f'avatar-man-1_hunyuan_{final_texture_label}_delivery'
if delivery.exists(): shutil.rmtree(delivery)
delivery.mkdir(parents=True, exist_ok=True)
shutil.copy2(out_dir / 'manifest.json', delivery / 'manifest.json')
shutil.copy2(final_glb, delivery / final_glb.name)
if shape_ok and final_glb != out_dir / 'shape.glb':
    shutil.copy2(out_dir / 'shape.glb', delivery / 'shape.glb')
zip_path = shutil.make_archive(str(out_dir / f'avatar-man-1_hunyuan_{final_texture_label}_delivery'), 'zip', root_dir=str(delivery))
print(json.dumps(manifest, indent=2))
print(f"\nOutputs in: {out_dir}")
print(f"Delivery ZIP: {zip_path}")
''' % SELECTED_IMAGE

script_path = '/kaggle/working/run_pipeline.py'
Path(script_path).write_text(pipeline_code)
print(f'Written {script_path} ({len(pipeline_code)} bytes)')
print(f'Executing with: {PYTHON}')
sys.stdout.flush()

result = subprocess.run([PYTHON, script_path], capture_output=False, timeout=3600)
print(f'\nPipeline exit code: {result.returncode}')

# ---- Verify outputs ----
out_dir = Path('/kaggle/working/outputs/avatar-man-1-hunyuan-fixed')
if out_dir.exists():
    print(f'\n=== Outputs in {out_dir} ===')
    for p in sorted(out_dir.iterdir()):
        size = p.stat().st_size
        print(f'  {p.name} ({size:,} bytes)')
else:
    print('Output directory not found')
